# Week 4, Part 2
## Feature Selection cont'd: Interactions, Outliers

Two datasets:

1. **Palmer penguins** 
2. **Bike rentals** 

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor, OLSInfluence

# Hello again Penguins 
peng = pd.read_csv("../data/penguins.csv").dropna(
    subset=["flipper_length_mm", "body_mass_g", "bill_length_mm", "bill_depth_mm"]
).reset_index(drop=True)

# and bikes!
bike = pd.read_csv("../data/bike-sharing-daily.csv", parse_dates=["dteday"])

## Interaction Terms

In [ ]:
## Let's build on what we did last class and dig further into species as a predictor

additive    = smf.ols("body_mass_g ~ flipper_length_mm + C(species)", data=peng).fit()
interaction = smf.ols("body_mass_g ~ flipper_length_mm * C(species)", data=peng).fit()

print(additive.summary())

In [ ]:
print(interaction.summary())

In [ ]:
# Let's plot this 

species = sorted(peng.species.unique())
palette = dict(zip(species, sns.color_palette(n_colors=len(species))))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
grid = np.linspace(peng.flipper_length_mm.min(), peng.flipper_length_mm.max(), 50)

for ax, model, title in [(axes[0], additive, "additive: shared slope"),
                         (axes[1], interaction, "interaction: slope per species")]:
    sns.scatterplot(data=peng, x="flipper_length_mm", y="body_mass_g",
                    hue="species", hue_order=species, palette=palette,
                    alpha=0.5, s=18, ax=ax, legend=(ax is axes[1]))
    for sp in species:
        pred = model.predict(pd.DataFrame({"flipper_length_mm": grid, "species": sp}))
        ax.plot(grid, pred, lw=2, color=palette[sp])
    ax.set_title(title)
plt.tight_layout()

In [ ]:
# Now let's use a nested F-test to see which model is truly better. 
compare = sm.stats.anova_lm(additive, interaction)
print(compare)

In [ ]:
# Let's now look at centering the data and see what that does to the model
# Note: Centering is a form of feature transformation!
# Note #2: This does not change the fit of the model!

peng["flipper_c"] = peng.flipper_length_mm - peng.flipper_length_mm.mean()
peng["bill_len_c"] = peng.bill_length_mm - peng.bill_length_mm.mean()

add_ctr    = smf.ols("body_mass_g ~ flipper_c + C(species)", data=peng).fit()
interact_ctr = smf.ols("body_mass_g ~ flipper_c * C(species)", data=peng).fit()

In [ ]:
# Let's compare centered and non-centered data 

print(pd.DataFrame({"raw": interaction.params, "centered": interact_ctr.params.values}).round(3))
print(f"\nR^2: raw {interaction.rsquared:.6f}   centered {interact_ctr.rsquared:.6f}   (identical model)")

In [ ]:
# Let's look at VIF and multicollinearity

def vif_table(model):
    # skip the intercept column (index 0)
    X = model.model.exog
    return [variance_inflation_factor(X, i) for i in range(1, X.shape[1])]

print(pd.DataFrame({"raw": vif_table(interaction), "centered": vif_table(interact_ctr)},
                   index=interact_ctr.model.exog_names[1:]).round(2))

In [ ]:
print(interact_ctr.summary())

In [ ]:
compare_ctr = sm.stats.anova_lm(add_ctr, interact_ctr)
print(compare_ctr)

In [ ]:
# Now let's look at the bike data
bike_additive = smf.ols("cnt ~ temp + C(workingday) + hum + windspeed", data=bike).fit()
bike_int = smf.ols("cnt ~ temp * C(workingday) + hum + windspeed", data=bike).fit()

In [ ]:
bike_compare = sm.stats.anova_lm(bike_additive, bike_int)
print(bike_compare)

In [ ]:
print(bike_int.summary())

## Outliers and High Leverage Values

In [ ]:
# Three different things. Fit a working model on the bike data first.

model = smf.ols("cnt ~ temp + hum + windspeed + C(season) + C(weathersit)", data=bike).fit()
infl = OLSInfluence(model) # Class that calculates outlier and influence measures for an OLS result

n, p = int(model.nobs), model.df_model
print(f"n = {n}, p = {p:.0f} ")
print("----------------------")
print(model.summary())


In [ ]:
# Let's plot fitted vs residuals
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(model.fittedvalues, model.resid, alpha=0.5)
ax.axhline(0, color="red", linestyle="--")
ax.set_xlabel("Fitted values")
ax.set_ylabel("Residuals")
ax.set_title("Residuals vs. Fitted")
plt.show()

In [ ]:
bike.cnt

In [ ]:
# Now let's build a dataframe calculating metrics for residuals, leverage, and cooks_d
diag = pd.DataFrame({
    "date":      bike.dteday,
    "cnt":       bike.cnt,
    "fitted":    model.fittedvalues,
    "residuals": model.resid,
    "student_resid": infl.resid_studentized_external,
    "leverage":  infl.hat_matrix_diag,
    "cooks_d":   infl.cooks_distance[0],
})

diag.head(10)

In [ ]:
# Now let's look at the cutoff values for this data, based on the formulas we looked at in the slides

lev_cut, cook_cut = 2 * (p + 1) / n, 4 / n
print(f"n = {n}, p = {p:.0f}   leverage cutoff = {lev_cut:.4f}   Cook's D cutoff = {cook_cut:.4f}")

# Count of observations that get flagged by each diagnostic
print(f"\noutliers   (|studentized resid| > 3): {(diag.student_resid.abs() > 3).sum()}")
print(f"high leverage (h > 2(p+1)/n):        {(diag.leverage > lev_cut).sum()}")
print(f"influential   (Cook's D > 4/n):      {(diag.cooks_d > cook_cut).sum()}")

In [ ]:
# Which point is the outlier
diag[diag.student_resid.abs() > 3]


In [ ]:
# Which are points of high leverage
diag[diag.leverage > lev_cut]

In [ ]:
# Which are points that are influential
diag[diag.cooks_d > cook_cut]

In [ ]:
# Plot all three at once: studentized residual vs. leverage, colored by Cook's D cutoff.
infl = diag.cooks_d > cook_cut

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(diag.leverage[~infl], diag.student_resid[~infl], s=20, alpha=0.4,
           color="tab:blue", label=f"Cook's D ≤ 4/n")
ax.scatter(diag.leverage[infl], diag.student_resid[infl], s=40, alpha=0.9,
           color="tab:orange", label=f"Cook's D > 4/n ({infl.sum()})")
ax.axhline(0, color="gray", lw=1)
for y in (-3, 3):
    ax.axhline(y, color="red", ls="--", lw=1)
ax.axvline(lev_cut, color="red", ls="--", lw=1)
ax.set(xlabel="leverage (h_ii)", ylabel="studentized residual",
       title="influence plot -- colored by Cook's D cutoff")
ax.legend()

for _, r in diag.nlargest(5, "cooks_d").iterrows():
    ax.annotate(r.date.strftime("%Y-%m-%d"), (r.leverage, r.student_resid),
                fontsize=8, xytext=(6, 4), textcoords="offset points")
plt.tight_layout()

In [ ]:
# Impact of removing influential points
keep = diag.cooks_d <= cook_cut
trimmed = smf.ols("cnt ~ temp + hum + windspeed + C(season) + C(weathersit)",
                  data=bike[keep.values]).fit()

print(pd.DataFrame({
    "all data": model.params,
    "influential dropped": trimmed.params,
    "% change": 100 * (trimmed.params - model.params) / model.params.abs(),
}).round(2))
print(f"\ndropped {(~keep).sum()} of {n} observations")
print(f"R^2: {model.rsquared:.4f} -> {trimmed.rsquared:.4f}")